In [1]:
# ==============================================================================
# 1. ENVIRONMENT & DEPENDENCIES
# ==============================================================================
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl evaluate accelerate peft bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-2c5ybmc1/unsloth_5c847a836ca643f49a77caaac9b18112
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-2c5ybmc1/unsloth_5c847a836ca643f49a77caaac9b18112
  Resolved https://github.com/unslothai/unsloth.git to commit cf97faed9ff73af5e46c18a21efdad24ded876dc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.8 MB/s eta 0:00:00

In [2]:
import os
import kagglehub
import json
import glob
import gc
import torch
import numpy as np
from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import DataCollatorForLanguageModeling
from trl import SFTTrainer, SFTConfig

downloaded_path = kagglehub.dataset_download('adhamashraf202200953/technical-parsed-questions')
print('Dataset Download Complete. Path:', downloaded_path)

DATA_FOLDER = downloaded_path
OUTPUT_DIR = "/kaggle/working/outputs/UNSLOTH_LLAMA3_1_8B"

gc.collect()
torch.cuda.empty_cache()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Dataset Download Complete. Path: /kaggle/input/datasets/adhamashraf202200953/technical-parsed-questions


In [3]:
# ==============================================================================
# 2. DATA LOADING & PARSING
# ==============================================================================
def load_and_parse_custom_jsonl(folder_path):
    all_records = []
    search_pattern = os.path.join(folder_path, "*.jsonl")
    jsonl_files = glob.glob(search_pattern)
    
    for file_path in jsonl_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                try:
                    record = json.loads(line)
                    if all(k in record for k in ["context", "scenario_context", "question", "model_answer"]):
                        all_records.append({
                            "context": record["context"],
                            "scenario_context": record["scenario_context"],
                            "question": record["question"],
                            "model_answer": record["model_answer"]
                        })
                except:
                    continue
    return all_records

raw_parsed_data = load_and_parse_custom_jsonl(DATA_FOLDER)
print(f"Successfully loaded {len(raw_parsed_data)} samples.")
dataset = Dataset.from_list(raw_parsed_data)

Successfully loaded 2190 samples.


In [4]:
max_seq_length = 512 
dtype = None          
load_in_4bit = True   

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-llama-3.1-8b-instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

llama3_prompt_style = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert technical interviewer. Based on the given textbook context, generate a professional workplace scenario context, an analytical interview question, and a detailed model answer.<|eot_id|><|start_header_id|>user<|end_header_id|>

Textbook Context:
{CONTEXT}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Scenario: {SCENARIO}
Question: {QUESTION}
Model Answer: {ANSWER}<|eot_id|>"""

==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
def formatting_prompts_func(examples):
    contexts = examples["context"]
    scenarios = examples["scenario_context"]
    questions = examples["question"]
    answers = examples["model_answer"]
    
    texts = []
    for c, s, q, a in zip(contexts, scenarios, questions, answers):
        text = llama3_prompt_style.format(CONTEXT=c, SCENARIO=s, QUESTION=q, ANSWER=a)
        texts.append(text)
        
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=max_seq_length,
        padding=False 
    )
    return tokenized

tokenized_dataset = dataset.map(
    formatting_prompts_func, 
    batched = True, 
    load_from_cache_file = False, 
    remove_columns = ["context", "scenario_context", "question", "model_answer"])

split_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=42)

training_args = SFTConfig(
    per_device_train_batch_size = 1,  
    per_device_eval_batch_size = 1,   
    gradient_accumulation_steps = 16, 
    warmup_ratio = 0.05,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    eval_strategy = "steps",
    eval_steps = 25,
    optim = "paged_adamw_8bit", 
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 42,
    output_dir = OUTPUT_DIR,
    report_to = "none",
    save_strategy = "steps",
    save_steps = 25,
    save_total_limit = 1,
    average_tokens_across_devices = False,
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = split_dataset["train"],
    eval_dataset = split_dataset["test"],
    args = training_args,
)

# ==============================================================================
# 7. EXECUTE FINE-TUNING & EXPORT
# ==============================================================================
print("Booting Unsloth Training engine for Llama 3.1...")
trainer.train()

final_adapter_path = "/kaggle/working/UNSLOTH_LLAMA3_1_8B_LORA"
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"LoRA adapters successfully exported to: {final_adapter_path}")

os.system(f'zip -r "/kaggle/working/UNSLOTH-LLAMA3-1-8B-LORA.zip" "{final_adapter_path}"')
print("Artifact zipped and ready for export.")

Map:   0%|          | 0/2190 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Booting Unsloth Training engine for Llama 3.1...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,080 | Num Epochs = 3 | Total steps = 390
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
25,1.380403,1.382850
50,1.278586,1.297064
75,1.297589,1.269101
100,1.230600,1.249299
125,1.289324,1.236409
150,1.130965,1.232997
175,1.115840,1.230313
200,1.100619,1.220908
225,1.115327,1.213653
250,1.128220,1.209033


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_LLAMA3_1_8B/checkpoint-25/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_LLAMA3_1_8B/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_LLAMA3_1_8B/checkpoint-75/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_LLAMA3_1_8B/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_LLAMA3_1_8B/checkpoint-125/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_LLAMA3_1_8B/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_LLAMA3_1_8B/checkpoint-175/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outpu

LoRA adapters successfully exported to: /kaggle/working/UNSLOTH_LLAMA3_1_8B_LORA
  adding: kaggle/working/UNSLOTH_LLAMA3_1_8B_LORA/ (stored 0%)
  adding: kaggle/working/UNSLOTH_LLAMA3_1_8B_LORA/README.md (deflated 65%)
  adding: kaggle/working/UNSLOTH_LLAMA3_1_8B_LORA/chat_template.jinja (deflated 72%)
  adding: kaggle/working/UNSLOTH_LLAMA3_1_8B_LORA/tokenizer.json (deflated 85%)
  adding: kaggle/working/UNSLOTH_LLAMA3_1_8B_LORA/tokenizer_config.json (deflated 96%)
  adding: kaggle/working/UNSLOTH_LLAMA3_1_8B_LORA/adapter_config.json (deflated 57%)
  adding: kaggle/working/UNSLOTH_LLAMA3_1_8B_LORA/adapter_model.safetensors (deflated 7%)
Artifact zipped and ready for export.
